# Huấn luyện YOLO11n và YOLO26n trên Colab (Nhóm 14)
Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy.
Notebook chỉ gọi lại các script trong `src/`, mọi tham số nằm ở `configs/config.yaml`.

## 1. Kết nối Drive và lấy mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Thư mục trên Drive chứa data/yolo (đã nén hoặc không nén) và nơi lưu kết quả
DRIVE_YOLO = '/content/drive/MyDrive/yolo'                      # SỬA nếu thư mục của bạn nằm chỗ khác
DRIVE_OUT  = '/content/drive/MyDrive/NHOM_14_ket_qua_yolo'      # kết quả huấn luyện sẽ chép về đây
REPO_URL   = 'https://github.com/hoangtrongkhoi050515/Product-recognition-in-supermarkets.git'
BRANCH     = 'claude/project-thread-bfomrh'                     # đổi thành 'main' sau khi PR được merge

In [ ]:
%cd /content
!rm -rf repo
!git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd /content/repo
!pip install -q "ultralytics>=8.4" pyyaml pandas

## 2. Chép dữ liệu về ổ Colab (đọc nhanh hơn đọc trực tiếp từ Drive) và sinh lại `data.yaml`
`data.yaml` trên máy Windows chứa đường dẫn `K:\...` nên phải sinh lại cho Colab.

In [ ]:
!mkdir -p data
!cp -r {DRIVE_YOLO} data/yolo
!rm -f data/yolo/data.yaml
!python -m src.data.split_dataset --yaml-only
!ls data/yolo/images/train | wc -l ; ls data/yolo/images/val | wc -l ; ls data/yolo/images/test | wc -l

## 3. Chạy thử 1 epoch trên 5% dữ liệu (kiểm tra mọi thứ chạy được, mất vài phút)

In [ ]:
!python -m src.detection.train --model yolo11n --epochs 1 --fraction 0.05

## 4. Huấn luyện thật (100 epoch, dừng sớm sau 20 epoch không cải thiện)

In [ ]:
!python -m src.detection.train --model yolo11n

In [ ]:
!python -m src.detection.train --model yolo26n

## 5. Đánh giá trên tập Validation và Test
Chỉ báo cáo kết quả tập **Test** ở Chương 3; tập Validation dùng để chọn mô hình.

In [ ]:
for m in ['yolo11n', 'yolo26n']:
    for s in ['val', 'test']:
        !python -m src.detection.evaluate --model {m} --split {s}

In [ ]:
import pandas as pd
for m in ['yolo11n', 'yolo26n']:
    print(m)
    display(pd.read_csv(f'Logs/detection/{m}_test/per_class.csv'))

## 6. Chép kết quả về Drive (Colab xoá sạch khi hết phiên)

In [ ]:
!mkdir -p {DRIVE_OUT}
!cp -r models {DRIVE_OUT}/
!cp -r Logs/detection {DRIVE_OUT}/
print('Đã chép về', DRIVE_OUT)